In [13]:
# colab setup
colab = False

In [ ]:
%load_ext autoreload
%autoreload 2
if colab:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd drive/MyDrive/ecg_arrhythmia/
    !pip install wfdb wget numpy pandas scipy scikit-learn tensorflow matplotlib seaborn PyWavelets
    !pip install --upgrade wfdb
    !pip install neurokit2
from hyperparams import *
from tasks import *
from plot import *
from model import *
from sklearn.metrics import classification_report
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedGroupKFold
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler

gridsearch_path = './gridsearch_seed/'
if not os.path.exists(gridsearch_path):
    os.makedirs(gridsearch_path)
resfile_name = 'splitseed0-39_withoutgroup.csv' 


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
mitdb, pwave = get_records(mitdb_dir, pwave_dir)

In [ ]:
# 전체 데이터를 저장할 리스트 초기화
all_segments = []
all_features = []
all_labels = []
all_records = []
i=0
# record별 전처리
for record in tqdm(mitdb):
        print(record)
        # load ECG signal & annotations
        sig, _ = load_ECG_signal(record)
        sig = np.squeeze(sig)

        annotations = load_ECG_annotations(record=record, dir=mitdb_dir, extension='atr')
        ann_sample = annotations.sample # rpeak 근처 annotation
        symbols = annotations.symbol  # label
        dct_symbols = dict(zip(ann_sample, symbols)) # key: rpeak idx, value: label
        dct_symbols = {k: v for k, v in dct_symbols.items() if v not in ex_labels} # filter symbols
        


        # rpeak detection
        rpeaks = get_rpeaks(sig, ecg_clean_method='biosppy', ecg_peaks_method='neurokit')
        adj_rpeaks, candid_rpeaks = adjust_rpeaks(sig, rpeaks)

        # sig normalization
        scaler = StandardScaler()
        sig = scaler.fit_transform(sig.reshape(-1, 1)).flatten()

        # feature extraction (HRV)
        features = nk.hrv(adj_rpeaks, sampling_rate=360, show=False).to_numpy()
        # features = scaler.fit_transform(features.reshape(-1, 1)).reshape(1,-1)
        features = np.repeat(features, len(adj_rpeaks) - 1, axis=0)
        all_features.append(features)



        # segmetation based on rpeaks
        segments = segmentation(sig, adj_rpeaks)
        # segments = segment_heartbeats(sig, adj_rpeaks)

        # label extraction & grouping
        labels = extract_labels(adj_rpeaks, dct_symbols)
        labels = list(map(group_labels, labels))
        labels = labels[1:]  # segmentation을 하기 때문에 마지막은 제거

        # split을 위한 record 인덱스 array 생성
        record_idx = np.array([record]*len(labels)) 

        # 데이터를 리스트에 추가
        all_labels.append(labels)
        all_records.append(record_idx)
        all_segments.append(segments)   
        # i+=1
        # if i == 1:
        #     break
        


    
    
x1 = np.concatenate(all_segments, axis=0)
x2 = np.concatenate(all_features, axis=0)
y = np.concatenate(all_labels, axis=0)
records = np.concatenate(all_records, axis=0)


print("Segments(x1) Shape:", x1.shape)
print("Extracted Features(x2) Shape:", x2.shape)
print("Labels(y) Shape:", y.shape)


118
124
111
104
107
220
109
207
223
212
100
217
114
228
106
203
219
214
201
215
221
103
232
222
115
119
102
202
209
108
117
208
213
234
121
116
205
122
123
105
233
113
101
112
210
231
230
200
Segments(x1) Shape: (109748, 300)
Extracted Features(x2) Shape: (109748, 91)
Labels(y) Shape: (109748,)


In [ ]:
## plotting
label_hist(y)
label_record_hist(y, records)
# get_label_record_ratio(y, records) # label별 record 비율 desc 으로 출력

In [ ]:
# metric_df를 담을 리스트 초기화
all_metrics = []
for seed in range(40):
    file_name = f'test_seed{seed}'
    # record 기준 data split
    # combine x1, x2 for split
    x1x2 = np.hstack((x1, x2))

    # data split stratified
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    # for fold, (train_val_idx, test_idx) in enumerate(sgkf.split(x1x2, y, groups=records)):
    for fold, (train_val_idx, test_idx) in enumerate(sgkf.split(x1x2, y)):

        break    
    x1x2_train_val = x1x2[train_val_idx]
    y_train_val = y[train_val_idx]
    records_train_val = records[train_val_idx]

    # Train/Validation 데이터를 다시 StratifiedGroupKFold로 나누기
    sgkf_val = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    # for train_idx, val_idx in sgkf_val.split(x1x2_train_val,y_train_val, groups=records_train_val):
    for train_idx, val_idx in sgkf_val.split(x1x2_train_val,y_train_val):

        break  # 첫 번째 split을 사용하여 Train/Validation 분할

    x1_train = x1x2_train_val[train_idx][:, :x1.shape[1]]
    x2_train = x1x2_train_val[train_idx][:, x1.shape[1]:]
    x1_val = x1x2_train_val[val_idx][:, :x1.shape[1]]
    x2_val = x1x2_train_val[val_idx][:, x1.shape[1]:]
    x1_test = x1x2[test_idx][:, :x1.shape[1]]
    x2_test = x1x2[test_idx][:, x1.shape[1]:]

    y_train = y_train_val[train_idx]
    y_val = y_train_val[val_idx]
    y_test = y[test_idx]


    # one-hot encoding
    y_train_oh, class_names = one_hot_encoder(y_train)
    y_val_oh, _ = one_hot_encoder(y_val)
    y_test_oh, _ = one_hot_encoder(y_test)
    # model initialization
    x1_shape = (x1_train.shape[1], 1)
    x2_shape = (x2_train.shape[1],)
    n_classes = y_train_oh.shape[1]
    model = CNNModel(x1_shape, x2_shape, n_classes)   
    # model training
    model.fit([x1_train, x2_train], y_train_oh, [x1_val, x2_val], y_val_oh, y_train)
    # model evaluation
    test_loss, test_accuracy = model.evaluate([x1_test,x2_test], y_test_oh)
    print(f"Test accuracy: {test_accuracy:.4f}")
    # prediction
    y_pred = model.predict([x1_test, x2_test])
    y_test = np.argmax(y_test_oh, axis=1)  

    # get precision, recall, f1-score, specificity
    precision, recall, f1, specificity = calc_metrics(y_test, y_pred, class_names)
    global_metrics = calc_global_metrics(y_test, y_pred)

    # report
    print(classification_report(y_test, y_pred, target_names=class_names))
    # get metric df
    metric_df = get_metric_df(precision, recall, f1, specificity, class_names)
    metric_df['seed'] = seed  # 각 seed 정보 추가
    print(metric_df)
    
    # 각 반복마다 생성된 metric_df를 리스트에 저장
    all_metrics.append(metric_df)


    # histogram
    plot_metric_hist(precision, recall, specificity, class_names)

    # # data export for colab
    # if colab:
    #     df_to_csv_colab(metric_df, file_name+'.csv')
    #     dict_to_json_colab(global_metrics, file_name+'.json')
    # else:
    #     metric_df.to_csv(gridsearch_path+file_name+'.csv', index=False)
    #     with open(gridsearch_path+file_name+'.json', 'w') as f:
    #         json.dump(global_metrics, f, indent=4)

# 모든 metric_df들을 하나의 데이터프레임으로 aggregation
aggregated_metrics = pd.concat(all_metrics, ignore_index=True)
aggregated_metrics.to_csv(gridsearch_path + resfile_name, index=False)

# 예시: seed별 또는 클래스별 평균 및 표준편차 계산하여 성능 비교
# summary_metrics = aggregated_metrics.groupby('seed').agg(['mean', 'std']).reset_index()
# print(summary_metrics)

In [30]:
aggregated_metrics.groupby('Class').mean()


,Precision,Recall,F1-Score,Specificity,seed
Class,,,,,
N,0.883724,0.929695,0.899173,0.433519,19.5
Q,0.394853,0.310840,0.305896,0.977539,19.5
S,0.139395,0.083032,0.070320,0.989043,19.5
V,0.562266,0.418714,0.408212,0.963957,19.5


In [31]:
aggregated_metrics.groupby('Class').max()

,Precision,Recall,F1-Score,Specificity,seed
Class,,,,,
N,0.990492,0.998478,0.973550,0.890287,39
Q,0.994966,0.997603,0.923635,1.000000,39
S,0.871795,0.557377,0.680000,1.000000,39
V,0.971609,0.957904,0.829320,0.999449,39


In [33]:
aggregated_metrics.groupby('Class').min()

,Precision,Recall,F1-Score,Specificity,seed
Class,,,,,
N,0.612021,0.481934,0.646124,0.020270,0
Q,0.000000,0.000000,0.000000,0.867665,0
S,0.000000,0.000000,0.000000,0.915789,0
V,0.000000,0.000000,0.000000,0.598472,0
